# HW4 — Data preparation and Feature Engineering

A signal is formed after a market close, and the earliest entry is the next market-session close. This experiment defines the planned exit as the close **60 market sessions after entry**. If the stock has no valid quote on that session, the exit moves to its next observed valid quote. The target uses those actual entry and exit prices.

This 60-session horizon differs from HW4's specified 90-calendar-day holding period, since I believe that 90 natural days can differ from the actual market experiencing sessions, leading to unstable and inaccurate training and prediction, thus we slightly modify the configuration.

In [1]:
# Find the repository and open the full homework dataset by path.
from pathlib import Path
import duckdb
import numpy as np
import pandas as pd

root = next((p for p in (Path.cwd(), *Path.cwd().parents) if (p / "Lectures/modeling/data/equities_top2000.duckdb").is_file()), None)
if root is None:
    raise FileNotFoundError("Missing Lectures/modeling/data/equities_top2000.duckdb")
db_path = root / "Lectures/modeling/data/equities_top2000.duckdb"
print("Database:", db_path)
print("Size (MB):", round(db_path.stat().st_size / 1024**2, 1))

Database: /Users/wangyisu/Desktop/Documents/pre-courses/Python for Financial Engineers/UCB-MFE-python-preprogram-2026-winter/Lectures/modeling/data/equities_top2000.duckdb
Size (MB): 314.8


In [2]:
# Inspect table sizes, schemas, and source examples before joining.
con = duckdb.connect(str(db_path), read_only=True)
tables = ["prices", "fundamentals", "universe", "universe_membership",
          "calendar", "benchmark", "dividends"]
display(pd.DataFrame([
    {"table": name, "rows": con.execute(f"SELECT COUNT(*) FROM {name}").fetchone()[0]}
    for name in tables
]))

for name in ["prices", "fundamentals", "universe",
             "universe_membership", "calendar", "benchmark"]:
    print(f"\n{name}")
    display(con.execute(f"DESCRIBE {name}").fetchdf()[["column_name", "column_type"]])
    display(con.execute(f"SELECT * FROM {name} LIMIT 3").fetchdf())

,table,rows
0,prices,15839788
1,fundamentals,284198
2,universe,3849
3,universe_membership,195970
4,calendar,6726
5,benchmark,6726
6,dividends,19625



prices


,column_name,column_type
0,symbol,VARCHAR
1,date,DATE
2,open,DOUBLE
3,high,DOUBLE
4,low,DOUBLE
5,close,DOUBLE
6,adj_close,DOUBLE
7,volume,BIGINT
8,source,VARCHAR


,symbol,date,open,high,low,close,adj_close,volume,source
0,A,2000-01-03,53.06,53.18,45.39,48.51,40.30,4962189,fmp
1,A,2000-01-04,45.90,46.40,43.63,44.17,36.69,5058514,fmp
2,A,2000-01-05,44.64,44.64,40.64,42.03,34.92,6113793,fmp



fundamentals


,column_name,column_type
0,symbol,VARCHAR
1,fiscal_date,DATE
2,period,VARCHAR
3,filing_date,DATE
4,revenue,DOUBLE
5,gross_profit,DOUBLE
6,operating_income,DOUBLE
7,net_income,DOUBLE
8,ebitda,DOUBLE
9,eps,DOUBLE


... (9 outputs left out)

,date
0,2000-01-03
1,2000-01-04
2,2000-01-05



benchmark


,column_name,column_type
0,symbol,VARCHAR
1,date,DATE
2,close,DOUBLE
3,adj_close,DOUBLE
4,source,VARCHAR


,symbol,date,close,adj_close,source
0,^SP500TR,2000-01-03,2002.11,2002.11,fmp
1,^SP500TR,2000-01-04,1925.41,1925.41,fmp
2,^SP500TR,2000-01-05,1929.28,1929.28,fmp


## 2 Data Preparation

### 2.1 Market calendar and benchmark

The supplied calendar defines market sessions. `ENTRY_date` is the next calendar session after each signal date. We match the S&P 500 total-return benchmark to that calendar and check both missing and extra dates. The holding horizon is set to 60 calendar **sessions**, not 60 stock-price rows.

In [3]:
# Build the shared market-session calendar and check benchmark coverage.
HOLDING_SESSIONS = 60

calendar_df = con.execute(
    "SELECT date AS CAL_date FROM calendar ORDER BY date"
).fetchdf()
calendar_df["CAL_date"] = pd.to_datetime(calendar_df["CAL_date"])
calendar_df["ENTRY_date"] = calendar_df["CAL_date"].shift(-1)

benchmark_raw = con.execute("""
    SELECT date AS BENCH_date, adj_close AS BENCH_adj_close
    FROM benchmark WHERE symbol = '^SP500TR' ORDER BY date
""").fetchdf()
benchmark_raw["BENCH_date"] = pd.to_datetime(benchmark_raw["BENCH_date"])
benchmark_df = calendar_df[["CAL_date"]].merge(
    benchmark_raw, left_on="CAL_date", right_on="BENCH_date",
    how="left", validate="one_to_one"
).drop(columns="BENCH_date").rename(columns={"CAL_date": "BENCH_date"})

benchmark_extra = (~benchmark_raw["BENCH_date"].isin(calendar_df["CAL_date"])).sum()
benchmark_invalid = (~np.isfinite(benchmark_df["BENCH_adj_close"]) |
                     benchmark_df["BENCH_adj_close"].le(0)).sum()
print("Market sessions:", len(calendar_df))
print("Benchmark dates outside calendar:", int(benchmark_extra))
print("Calendar dates without a valid benchmark:", int(benchmark_invalid))
display(calendar_df.head())
display(benchmark_df.head())
if benchmark_extra or benchmark_invalid:
    raise ValueError("Benchmark and market calendar do not align")

Market sessions: 6726
Benchmark dates outside calendar: 0
Calendar dates without a valid benchmark: 0


,CAL_date,ENTRY_date
0,2000-01-03,2000-01-04
1,2000-01-04,2000-01-05
2,2000-01-05,2000-01-06
3,2000-01-06,2000-01-07
4,2000-01-07,2000-01-10


,BENCH_date,BENCH_adj_close
0,2000-01-03,2002.11
1,2000-01-04,1925.41
2,2000-01-05,1929.28
3,2000-01-06,1931.51
4,2000-01-07,1983.83


### 2.2 Available quarterly statements

We retain the quarterly statements' numeric source fields, fiscal and filing dates, period start when supplied, and reported currency. A statement becomes available on the calendar day after filing. At each signal close, we will attach the most recent fiscal quarter already available; a late filing for an older quarter cannot replace a newer known quarter.

These are source values, not model features yet. The source does not provide reliable original filing vintages, and many reported currencies are missing. We therefore do not assume that every raw monetary amount is directly comparable across stocks. Dividends are excluded: their announcement or first-available dates are not supplied, and adjusted-price returns must not receive dividends a second time.

In [4]:
# Read every numeric financial field, plus the dates and metadata needed to use it.
fund_schema = con.execute("DESCRIBE fundamentals").fetchdf()
fund_schema["column_type"] = fund_schema["column_type"].str.upper()
numeric_cols = fund_schema.loc[
    fund_schema["column_type"].str.match(
        r"^(?:U?(?:TINYINT|SMALLINT|INTEGER|BIGINT|HUGEINT)|FLOAT|DOUBLE|REAL|DECIMAL|NUMERIC)"
    ), "column_name"
].tolist()
identifier_cols = {"id", "cik", "company_id", "security_id", "permno", "gvkey"}
numeric_cols = [col for col in numeric_cols if col not in identifier_cols]

base_cols = ["symbol", "fiscal_date", "filing_date", "period", "reported_currency"]
if "period_start" in fund_schema["column_name"].values:
    base_cols.append("period_start")
fund_cols = list(dict.fromkeys(base_cols + numeric_cols))
fund_sql = ", ".join(f'"{col}"' for col in fund_cols)
financial_raw = con.execute(
    f"SELECT {fund_sql} FROM fundamentals"
).fetchdf().add_prefix("FUN_")

for col in ["FUN_fiscal_date", "FUN_filing_date", "FUN_period_start"]:
    if col in financial_raw:
        financial_raw[col] = pd.to_datetime(financial_raw[col])

print("Retained numeric financial fields:", len(numeric_cols))
print(numeric_cols)
print("Source financial rows:", len(financial_raw))
display(financial_raw.head(3))

Retained numeric financial fields: 38
['revenue', 'gross_profit', 'operating_income', 'net_income', 'ebitda', 'eps', 'eps_diluted', 'operating_cash_flow', 'capex', 'free_cash_flow', 'total_assets', 'total_debt', 'total_equity', 'cash_and_equivalents', 'market_cap', 'enterprise_value', 'shares_outstanding', 'stock_based_compensation', 'common_stock_issued', 'common_stock_repurchased', 'dividends_paid', 'cost_of_revenue', 'rd_expense', 'interest_expense', 'income_tax_expense', 'dep_amort', 'pretax_income', 'change_in_working_capital', 'total_current_assets', 'total_current_liabilities', 'inventory', 'net_receivables', 'ppe_net', 'retained_earnings', 'total_liabilities', 'short_term_debt', 'long_term_debt', 'goodwill_and_intangibles']
Source financial rows: 284198


,FUN_symbol,FUN_fiscal_date,FUN_filing_date,FUN_period,FUN_reported_currency,FUN_period_start,FUN_revenue,FUN_gross_profit,FUN_operating_income,FUN_net_income,...,FUN_total_current_assets,FUN_total_current_liabilities,FUN_inventory,FUN_net_receivables,FUN_ppe_net,FUN_retained_earnings,FUN_total_liabilities,FUN_short_term_debt,FUN_long_term_debt,FUN_goodwill_and_intangibles
0,A,2000-01-31,2000-03-15,quarter,None,NaT,2.246000e+09,1.086000e+09,171000000.0,131000000.0,...,4.982000e+09,2.067000e+09,1.567000e+09,1.445000e+09,1.408000e+09,131000000.0,2.621000e+09,443000000.0,554000000.0,0.0
1,A,2000-04-30,2000-06-12,quarter,None,NaT,2.485000e+09,7.480000e+08,214000000.0,166000000.0,...,5.057000e+09,2.186000e+09,1.622000e+09,1.889000e+09,1.453000e+09,297000000.0,2.679000e+09,98000000.0,0.0,0.0
2,A,2000-07-31,2000-09-01,quarter,None,NaT,2.670000e+09,8.250000e+08,210000000.0,155000000.0,...,5.344000e+09,2.510000e+09,1.762000e+09,2.167000e+09,1.581000e+09,452000000.0,2.925000e+09,129000000.0,0.0,0.0


In [5]:
# Keep usable quarterly filings and prevent an older fiscal period from taking over.
financial_df = financial_raw.loc[
    financial_raw["FUN_period"].eq("quarter") &
    financial_raw["FUN_filing_date"].gt(financial_raw["FUN_fiscal_date"])
].copy()
financial_df["FUN_available_date"] = financial_df["FUN_filing_date"] + pd.Timedelta(days=1)
financial_df = financial_df.sort_values(
    ["FUN_symbol", "FUN_available_date", "FUN_fiscal_date"]
)
latest_fiscal = financial_df.groupby("FUN_symbol")["FUN_fiscal_date"].cummax()
financial_df = financial_df.loc[
    financial_df["FUN_fiscal_date"].eq(latest_fiscal)
].drop_duplicates(
    ["FUN_symbol", "FUN_available_date"], keep="last"
).drop(columns="FUN_period").reset_index(drop=True)

print("Usable quarterly filing events:", len(financial_df))
print("Excluded source rows:", len(financial_raw) - len(financial_df))
display(financial_df.loc[financial_df["FUN_symbol"].eq("AAPL")].head(3))

Usable quarterly filing events: 255218
Excluded source rows: 28980


,FUN_symbol,FUN_fiscal_date,FUN_filing_date,FUN_reported_currency,FUN_period_start,FUN_revenue,FUN_gross_profit,FUN_operating_income,FUN_net_income,FUN_ebitda,...,FUN_total_current_liabilities,FUN_inventory,FUN_net_receivables,FUN_ppe_net,FUN_retained_earnings,FUN_total_liabilities,FUN_short_term_debt,FUN_long_term_debt,FUN_goodwill_and_intangibles,FUN_available_date
920,AAPL,2000-01-01,2000-02-01,None,NaT,2.343000e+09,607000000.0,100000000.0,183000000.0,299000000.0,...,1.965000e+09,15000000.0,892000000.0,318000000.0,1.682000e+09,3.172000e+09,0.0,300000000.0,0.0,2000-02-02
921,AAPL,2000-04-01,2000-05-11,None,NaT,1.945000e+09,549000000.0,170000000.0,233000000.0,191000000.0,...,1.853000e+09,10000000.0,940000000.0,314000000.0,1.915000e+09,2.792000e+09,0.0,300000000.0,0.0,2000-05-12
922,AAPL,2000-07-01,2000-07-31,None,NaT,1.825000e+09,543000000.0,168000000.0,200000000.0,300000000.0,...,1.873000e+09,5000000.0,967000000.0,314000000.0,2.115000e+09,2.756000e+09,0.0,300000000.0,0.0,2000-08-01


### 2.3 Security descriptors, quarterly membership, and prices

Prices are the base of the stock history. We keep the security's sector, industry, country, and exchange from `universe`, while excluding current active status and full-history selection summaries. These descriptors are supplied as a snapshot, so their historical accuracy is a limitation.

Quarterly membership provides the selection date, rank, recorded shares, selection close, and capitalization proxy. Membership controls **new entries** for the quarter containing the next-session entry date. A security's price history remains available after it leaves the entry universe, because an existing holding may still need to be valued and sold.

In [6]:
# Load useful descriptors and the dated quarterly selection records.
universe_df = con.execute("""
    SELECT symbol AS UNIV_symbol, sector AS UNIV_sector,
           industry AS UNIV_industry, country AS UNIV_country,
           exchange AS UNIV_exchange
    FROM universe
""").fetchdf()

membership_df = con.execute("""
    SELECT symbol AS MEM_symbol, quarter_start AS MEM_quarter_start,
           quarter_end AS MEM_quarter_end, selection_date AS MEM_selection_date,
           selection_close AS MEM_selection_close,
           selection_shares AS MEM_selection_shares,
           market_cap_proxy AS MEM_market_cap_proxy,
           universe_rank AS MEM_universe_rank
    FROM universe_membership
""").fetchdf()
for col in ["MEM_quarter_start", "MEM_quarter_end", "MEM_selection_date"]:
    membership_df[col] = pd.to_datetime(membership_df[col])

print("Universe:", universe_df.shape, "Membership:", membership_df.shape)
display(universe_df.head(3))
display(membership_df.head(3))

Universe: (3849, 5) Membership: (195970, 8)


,UNIV_symbol,UNIV_sector,UNIV_industry,UNIV_country,UNIV_exchange
0,A,Healthcare,Medical - Diagnostics & Research,US,NYSE
1,AA,Basic Materials,Aluminum,US,NYSE
2,AAC,Financial Services,Financial - Conglomerates,US,NYSE


,MEM_symbol,MEM_quarter_start,MEM_quarter_end,MEM_selection_date,MEM_selection_close,MEM_selection_shares,MEM_market_cap_proxy,MEM_universe_rank
0,SEEL,2000-01-01,2000-03-31,1999-12-31,71243520.00,1.000000e+04,7.124352e+11,1
1,MSFT,2000-01-01,2000-03-31,1999-12-31,58.38,1.105400e+10,6.453325e+11,2
2,CSCO,2000-01-01,2000-03-31,1999-12-31,53.56,7.288000e+09,3.903453e+11,3


In [7]:
# Keep the source OHLC, adjusted close, and volume for the full stock history.
prices_df = con.execute("""
    SELECT symbol AS PRICE_symbol, date AS PRICE_date,
           open AS PRICE_open, high AS PRICE_high, low AS PRICE_low,
           close AS PRICE_close, adj_close AS PRICE_adj_close,
           volume AS PRICE_volume
    FROM prices ORDER BY symbol, date
""").fetchdf()
prices_df["PRICE_date"] = pd.to_datetime(prices_df["PRICE_date"])
con.close()

duplicates = prices_df.duplicated(["PRICE_symbol", "PRICE_date"]).sum()
print("Price rows:", len(prices_df), "Duplicate stock-date keys:", int(duplicates))
display(prices_df.head(3))
if duplicates:
    raise ValueError("Duplicate stock-date price keys need inspection")

Price rows: 15839788 Duplicate stock-date keys: 0


,PRICE_symbol,PRICE_date,PRICE_open,PRICE_high,PRICE_low,PRICE_close,PRICE_adj_close,PRICE_volume
0,A,2000-01-03,53.06,53.18,45.39,48.51,40.30,4962189
1,A,2000-01-04,45.90,46.40,43.63,44.17,36.69,5058514
2,A,2000-01-05,44.64,44.64,40.64,42.03,34.92,6113793


### 2.4 Join the stock history

We join calendar dates to prices, then attach the latest statement available at each signal close, security descriptors, and membership for the **entry date's quarter**. `flag_tradeable` marks a possible new-entry signal when the signal-date adjusted close is valid, a next market session exists, and that quarter's membership was already known.

All calendar-aligned price rows remain in `stock_df`, including rows with `flag_tradeable = 0`. This preserves historical lookbacks and quotes needed for positions already held. The flag does not promise that the stock will have an observed price at the next session's close.

In [8]:
# Align prices to the supplied calendar and show source rows outside it.
calendar_join_df = prices_df.merge(
    calendar_df, left_on="PRICE_date", right_on="CAL_date",
    how="left", validate="many_to_one", indicator="_calendar_match"
)
off_calendar_df = calendar_join_df.loc[
    calendar_join_df["_calendar_match"].eq("left_only")
].copy()
stock_df = calendar_join_df.loc[
    calendar_join_df["_calendar_match"].eq("both")
].drop(columns=["CAL_date", "_calendar_match"]).copy()
del calendar_join_df, prices_df

print("Off-calendar price rows:", len(off_calendar_df))
print("Calendar-aligned stock rows:", len(stock_df))
display(off_calendar_df[["PRICE_symbol", "PRICE_date"]].head())
display(stock_df.head(3))

Off-calendar price rows: 28
Calendar-aligned stock rows: 15839760


,PRICE_symbol,PRICE_date
2075430,BMNR,2024-11-28
2075526,BMNR,2025-04-18
2075552,BMNR,2025-05-26
4514883,DYNT,2024-09-02
4514973,DYNT,2025-01-09


,PRICE_symbol,PRICE_date,PRICE_open,PRICE_high,PRICE_low,PRICE_close,PRICE_adj_close,PRICE_volume,ENTRY_date
0,A,2000-01-03,53.06,53.18,45.39,48.51,40.30,4962189,2000-01-04
1,A,2000-01-04,45.90,46.40,43.63,44.17,36.69,5058514,2000-01-05
2,A,2000-01-05,44.64,44.64,40.64,42.03,34.92,6113793,2000-01-06


In [9]:
# Match each stock with its latest statement available at the signal close.
stock_df["PRICE_date"] = stock_df["PRICE_date"].astype("datetime64[ns]")
financial_df["FUN_available_date"] = financial_df["FUN_available_date"].astype("datetime64[ns]")
stock_df = pd.merge_asof(
    stock_df.sort_values("PRICE_date"),
    financial_df.sort_values("FUN_available_date"),
    left_on="PRICE_date", right_on="FUN_available_date",
    left_by="PRICE_symbol", right_by="FUN_symbol",
    direction="backward"
).drop(columns="FUN_symbol")
stock_df["FUN_filing_age_days"] = (
    stock_df["PRICE_date"] - stock_df["FUN_filing_date"]
).dt.days
stock_df["FUN_fiscal_age_days"] = (
    stock_df["PRICE_date"] - stock_df["FUN_fiscal_date"]
).dt.days

print("After financial join:", stock_df.shape)
display(stock_df[["PRICE_symbol", "PRICE_date", "FUN_fiscal_date",
                  "FUN_filing_date", "FUN_available_date",
                  "FUN_revenue", "FUN_net_income",
                  "FUN_filing_age_days", "FUN_fiscal_age_days"]].head())

After financial join: (15839760, 54)


,PRICE_symbol,PRICE_date,FUN_fiscal_date,FUN_filing_date,FUN_available_date,FUN_revenue,FUN_net_income,FUN_filing_age_days,FUN_fiscal_age_days
0,A,2000-01-03,NaT,NaT,NaT,NaN,NaN,NaN,NaN
1,SYBT,2000-01-03,NaT,NaT,NaT,NaN,NaN,NaN,NaN
2,LOW,2000-01-03,NaT,NaT,NaT,NaN,NaN,NaN,NaN
3,LNW,2000-01-03,NaT,NaT,NaT,NaN,NaN,NaN,NaN
4,SYK,2000-01-03,NaT,NaT,NaT,NaN,NaN,NaN,NaN


In [10]:
# Add source descriptors and membership for the next entry session's quarter.
stock_df = stock_df.merge(
    universe_df, left_on="PRICE_symbol", right_on="UNIV_symbol",
    how="left", validate="many_to_one"
).drop(columns="UNIV_symbol")
stock_df["ENTRY_quarter"] = stock_df["ENTRY_date"].dt.to_period("Q").dt.start_time
stock_df = stock_df.merge(
    membership_df,
    left_on=["PRICE_symbol", "ENTRY_quarter"],
    right_on=["MEM_symbol", "MEM_quarter_start"],
    how="left", validate="many_to_one"
).drop(columns=["MEM_symbol", "MEM_quarter_start"])

valid_signal_price = (
    np.isfinite(stock_df["PRICE_adj_close"]) &
    stock_df["PRICE_adj_close"].gt(0)
)
known_member = (
    stock_df["MEM_selection_date"].notna() &
    stock_df["MEM_selection_date"].le(stock_df["PRICE_date"])
)
stock_df["flag_tradeable"] = (
    valid_signal_price & stock_df["ENTRY_date"].notna() & known_member
).astype(int)

print("Joined stock table:", stock_df.shape)
display(stock_df.loc[
    stock_df["PRICE_symbol"].eq("AAPL") &
    stock_df["PRICE_date"].between("2010-03-30", "2010-04-01"),
    ["PRICE_symbol", "PRICE_date", "ENTRY_date", "ENTRY_quarter",
     "FUN_filing_date", "UNIV_sector", "UNIV_industry",
     "MEM_selection_date", "MEM_selection_shares",
     "MEM_market_cap_proxy", "MEM_universe_rank", "flag_tradeable"]
])
display(stock_df["flag_tradeable"].value_counts().rename("rows").to_frame())

Joined stock table: (15839760, 66)


,PRICE_symbol,PRICE_date,ENTRY_date,ENTRY_quarter,FUN_filing_date,UNIV_sector,UNIV_industry,MEM_selection_date,MEM_selection_shares,MEM_market_cap_proxy,MEM_universe_rank,flag_tradeable
4401137,AAPL,2010-03-30,2010-03-31,2010-01-01,2010-01-25,Technology,Consumer Electronics,2009-12-31,2.560247e+10,1.927866e+11,9.0,1
4403048,AAPL,2010-03-31,2010-04-01,2010-04-01,2010-01-25,Technology,Consumer Electronics,2010-03-31,2.575392e+10,2.160754e+11,8.0,1
4404075,AAPL,2010-04-01,2010-04-05,2010-04-01,2010-01-25,Technology,Consumer Electronics,2010-03-31,2.575392e+10,2.160754e+11,8.0,1


,rows
flag_tradeable,
1,12284888
0,3554872


### 2.5 Construct the 60-session target

For each eligible signal, entry is at the next **market session's observed** adjusted close. We count the entry session as position zero and schedule the exit 60 further market sessions later. This uses the common calendar even if that stock has missing quotes.

If the entry session lacks a valid stock quote, there was no purchase and its target remains missing. If the scheduled exit session lacks a valid quote, the observed exit moves to the stock's next valid quote. `scheduled_exit` records the planned session, `label_end` the actual observed exit, and `target` the gross adjusted-price return. End-of-data outcomes remain missing. These future-outcome columns must never be model inputs or used to screen prediction candidates.

In [11]:
# Use observed positive adjusted closes for entry and exit.
MAX_EXIT_DELAY = 5
valid_quote = np.isfinite(stock_df["PRICE_adj_close"]) & stock_df["PRICE_adj_close"].gt(0)
quotes_df = stock_df.loc[
    valid_quote, ["PRICE_symbol", "PRICE_date", "PRICE_adj_close"]
].copy()

# The entry is the next market session; a missing entry quote leaves no target.
labels_df = stock_df.loc[
    stock_df["flag_tradeable"].eq(1), ["PRICE_symbol", "ENTRY_date"]
].copy()
labels_df["_row"] = labels_df.index
labels_df = labels_df.reset_index(drop=True)
labels_df = labels_df.merge(
    quotes_df.rename(columns={
        "PRICE_date": "ENTRY_date", "PRICE_adj_close": "_entry_price"
    }),
    on=["PRICE_symbol", "ENTRY_date"], how="left", validate="many_to_one"
)

# Schedule the exit 60 market sessions after the actual entry session.
sessions = pd.DatetimeIndex(calendar_df["CAL_date"])
exit_pos = sessions.get_indexer(labels_df["ENTRY_date"]) + HOLDING_SESSIONS
labels_df["scheduled_exit"] = pd.NaT
in_range = exit_pos < len(sessions)
labels_df.loc[in_range, "scheduled_exit"] = sessions.take(exit_pos[in_range]).to_numpy()

# Find the first observed quote on or after the scheduled exit.
future = labels_df.loc[
    labels_df["_entry_price"].notna() & labels_df["scheduled_exit"].notna(),
    ["_row", "PRICE_symbol", "scheduled_exit", "_entry_price"]
].sort_values("scheduled_exit")
exit_quotes = quotes_df.rename(columns={
    "PRICE_date": "label_end", "PRICE_adj_close": "_exit_price"
}).sort_values("label_end")
observed = pd.merge_asof(
    future, exit_quotes,
    left_on="scheduled_exit", right_on="label_end",
    by="PRICE_symbol", direction="forward"
)

# A quote more than five market sessions late is not an observed 60-session outcome.
exit_delay = (
    sessions.get_indexer(observed["label_end"]) -
    sessions.get_indexer(observed["scheduled_exit"])
)
too_late = observed["label_end"].notna() & (exit_delay > MAX_EXIT_DELAY)
observed.loc[too_late, "label_end"] = pd.NaT
observed.loc[too_late, "_exit_price"] = np.nan

# Keep signal rows; write only observed, timely outcomes as labels.
stock_df["scheduled_exit"] = pd.NaT
stock_df["label_end"] = pd.NaT
stock_df["target"] = np.nan
stock_df.loc[labels_df["_row"], "scheduled_exit"] = labels_df["scheduled_exit"].to_numpy()
stock_df.loc[observed["_row"], "label_end"] = observed["label_end"].to_numpy()
stock_df.loc[observed["_row"], "target"] = (
    observed["_exit_price"] / observed["_entry_price"] - 1
).to_numpy()

print("Eligible signal rows:", len(labels_df))
print("No observed next-session entry:", int(labels_df["_entry_price"].isna().sum()))
print("Exits censored after five sessions:", int(too_late.sum()))
print("Observed completed targets:", int(stock_df["target"].notna().sum()))
display(stock_df.loc[
    stock_df["PRICE_symbol"].eq("AAPL") &
    stock_df["PRICE_date"].between("2012-08-30", "2012-09-05"),
    ["PRICE_symbol", "PRICE_date", "ENTRY_date",
     "scheduled_exit", "label_end", "target"]
])

Eligible signal rows: 12284888
No observed next-session entry: 7993
Exits censored after five sessions: 2830
Observed completed targets: 12120871


,PRICE_symbol,PRICE_date,ENTRY_date,scheduled_exit,label_end,target
5703130,AAPL,2012-08-30,2012-08-31,2012-11-29,2012-11-29,-0.110445
5706227,AAPL,2012-08-31,2012-09-04,2012-11-30,2012-11-30,-0.129557
5708136,AAPL,2012-09-04,2012-09-05,2012-12-03,2012-12-03,-0.121528
5709032,AAPL,2012-09-05,2012-09-06,2012-12-04,2012-12-04,-0.144122


### 2.6 Check the prepared data

We check row preservation, calendar and benchmark coverage, filing and membership timing, target availability, and delayed exits. Missing financial statements remain missing. A delayed observed exit is not a data fill: the later quote must actually exist.

Before building the backtester, we will review long quote gaps and define how an open position is valued during them. Its actual sales must follow the same 60-session schedule and observed-price rule used by these labels.

In [12]:
# Check coverage, timing, missing source values, and delayed exits.
checks = pd.Series({
    "stock_rows": len(stock_df),
    "off_calendar_source_rows": len(off_calendar_df),
    "duplicate_stock_dates": int(stock_df.duplicated(["PRICE_symbol", "PRICE_date"]).sum()),
    "stock_dates_outside_calendar": int((~stock_df["PRICE_date"].isin(sessions)).sum()),
    "calendar_dates_without_any_stock": int((~calendar_df["CAL_date"].isin(stock_df["PRICE_date"])).sum()),
    "benchmark_dates_outside_calendar": int((~benchmark_df["BENCH_date"].isin(sessions)).sum()),
    "calendar_dates_without_benchmark": int((~calendar_df["CAL_date"].isin(benchmark_df["BENCH_date"])).sum()),
    "filings_used_before_available": int(stock_df["FUN_available_date"].gt(stock_df["PRICE_date"]).sum()),
    "membership_used_before_selection": int(
        (stock_df["flag_tradeable"].eq(1) &
         stock_df["MEM_selection_date"].gt(stock_df["PRICE_date"])).sum()
    ),
    "tradeable_signal_rows": int(stock_df["flag_tradeable"].sum()),
    "completed_targets": int(stock_df["target"].notna().sum()),
    "targets_without_tradeable_signal": int(
        (stock_df["target"].notna() & stock_df["flag_tradeable"].eq(0)).sum()
    )
}, name="rows")
display(checks.to_frame())

source_cols = ["PRICE_adj_close", "PRICE_volume", "FUN_filing_date",
               "FUN_revenue", "FUN_net_income", "FUN_total_assets",
               "FUN_reported_currency", "UNIV_sector", "UNIV_industry",
               "MEM_selection_shares", "MEM_market_cap_proxy"]
display(stock_df[source_cols].isna().sum().rename("missing_rows").to_frame())

completed = stock_df["label_end"].notna()
delay_sessions = (
    sessions.get_indexer(stock_df.loc[completed, "label_end"]) -
    sessions.get_indexer(stock_df.loc[completed, "scheduled_exit"])
)
display(pd.Series({
    "on_scheduled_session": int((delay_sessions == 0).sum()),
    "delayed_1_to_5_sessions": int(((delay_sessions >= 1) & (delay_sessions <= 5)).sum()),
    "delayed_over_5_sessions": int((delay_sessions > 5).sum()),
    "longest_delay_sessions": int(delay_sessions.max()) if len(delay_sessions) else 0
}, name="completed_labels").to_frame())

display(stock_df[["PRICE_symbol", "PRICE_date", "ENTRY_date",
                  "scheduled_exit", "label_end", "target",
                  "flag_tradeable"]].tail(5))

,rows
stock_rows,15839760
off_calendar_source_rows,28
duplicate_stock_dates,0
stock_dates_outside_calendar,0
calendar_dates_without_any_stock,0
benchmark_dates_outside_calendar,0
calendar_dates_without_benchmark,0
filings_used_before_available,0
membership_used_before_selection,0
tradeable_signal_rows,12284888


,missing_rows
PRICE_adj_close,0
PRICE_volume,0
FUN_filing_date,748793
FUN_revenue,748983
FUN_net_income,748793
FUN_total_assets,775833
FUN_reported_currency,14965367
UNIV_sector,0
UNIV_industry,0
MEM_selection_shares,3554872


,completed_labels
on_scheduled_session,12114588
delayed_1_to_5_sessions,6283
delayed_over_5_sessions,0
longest_delay_sessions,5


,PRICE_symbol,PRICE_date,ENTRY_date,scheduled_exit,label_end,target,flag_tradeable
15839755,GBX,2026-09-30,NaT,NaT,NaT,NaN,0
15839756,ALMS,2026-09-30,NaT,NaT,NaT,NaN,0
15839757,CVM,2026-09-30,NaT,NaT,NaT,NaN,0
15839758,MLKN,2026-09-30,NaT,NaT,NaT,NaN,0
15839759,GCBC,2026-09-30,NaT,NaT,NaT,NaN,0


### 2.7 Sample coverage and missingness screening

We first report how many stock rows are eligible new-entry signals and how many also have an observed target. Column missingness is measured only among rows with `flag_tradeable = 1`.

Source columns missing in more than 5% of these rows are removed from the prepared `stock_df`: they provide too little consistent coverage for this analysis. The underlying source tables remain available for inspection. All other missing values are left unchanged until we have constructed and selected the model features. Signal dates, eligibility, and future-outcome columns are not screened as predictors.

In [13]:
# Report coverage relative to all stock rows.
tradeable = stock_df["flag_tradeable"].eq(1)
labeled = tradeable & stock_df["target"].notna()
n_all, n_tradeable = len(stock_df), int(tradeable.sum())

display(pd.DataFrame({
    "rows": [n_all, n_tradeable, int(labeled.sum())],
    "pct_of_all": [100, 100 * n_tradeable / n_all, 100 * labeled.sum() / n_all]
}, index=["All rows", "Tradeable rows", "Tradeable with target"]).round(2))

# Measure each column's missingness among tradeable signal rows.
missing_rows = []
for col in stock_df.columns:
    missing_count = int((stock_df[col].isna() & tradeable).sum())
    missing_rows.append({
        "column": col,
        "missing_rows": missing_count,
        "missing_pct": 100 * missing_count / n_tradeable
    })
missing_report = pd.DataFrame(missing_rows)

with pd.option_context("display.max_rows", None):
    display(missing_report.sort_values("missing_pct", ascending=False)
            .round({"missing_pct": 2}))

# Remove source columns above 5%; retain dates, eligibility, and labels.
source_col = missing_report["column"].str.startswith(("PRICE_", "FUN_", "UNIV_", "MEM_"))
drop_cols = missing_report.loc[
    source_col & missing_report["missing_pct"].gt(5), "column"
].tolist()
stock_df.drop(columns=drop_cols, inplace=True)

print("Removed columns:", drop_cols)
print("Prepared stock table:", stock_df.shape)

,rows,pct_of_all
All rows,15839760,100.00
Tradeable rows,12284888,77.56
Tradeable with target,12120871,76.52


,column,missing_rows,missing_pct
19,FUN_eps_diluted,12284246,99.99
12,FUN_period_start,12284007,99.99
27,FUN_market_cap,12016976,97.82
28,FUN_enterprise_value,12016976,97.82
11,FUN_reported_currency,11604713,94.46
33,FUN_dividends_paid,5438936,44.27
40,FUN_change_in_working_capital,396493,3.23
48,FUN_short_term_debt,376468,3.06
49,FUN_long_term_debt,376157,3.06
50,FUN_goodwill_and_intangibles,376157,3.06


Removed columns: ['FUN_reported_currency', 'FUN_period_start', 'FUN_eps_diluted', 'FUN_market_cap', 'FUN_enterprise_value', 'FUN_dividends_paid']
Prepared stock table: (15839760, 63)


## 3 Feature Engineering

### 3.1 Price and volume features

Features are calculated separately for each stock on the supplied market-session calendar. A missing stock quote remains missing. In particular, a one-session return is missing if the preceding market session has no valid quote; we do not calculate a return across that gap. All rolling windows below count **market sessions**, including the signal session.

**Price and volume adjustment.** For a date with positive raw and adjusted closes, the adjustment factor is `PRICE_adj_close / PRICE_close`. Multiplying raw open, high, and low by this factor gives adjusted OHLC prices. Dividing raw volume by the factor gives a synthetic adjusted-volume series. Because the price factor may reflect dividends as well as splits, synthetic adjusted volume is used only for statistical features; it is not the actual number of shares traded. Turnover uses raw volume and the quarterly membership's recorded shares instead.

The resulting feature groups are:

| Group | Input and calculation | Windows | Number of features |
|---|---|---:|---:|
| One-session change distribution | For adjusted close and synthetic adjusted volume, first calculate `current / previous_market_session - 1`. Calculate its rolling mean; for longer windows also calculate rolling standard deviation, skewness, and kurtosis. | Mean: 1, 10, 60, 240; other statistics: 10, 60, 240 | 26 |
| Current drawdown | `current_adjusted_close / highest_adjusted_close_in_window - 1`. This measures the current position below a recent high; it is not the maximum peak-to-trough drawdown anywhere inside the window. | 10, 60, 240 | 3 |
| RSI | Use the window's average positive and negative adjusted-close changes to measure the balance of recent gains and losses. This is a simple rolling-average RSI. | 10, 60 | 2 |
| MACD | Subtract the 26-session EMA from the 12-session EMA. Also calculate the difference between MACD and its 9-session EMA signal line. Divide both results by the current adjusted close. | 12/26/9 EMA | 2 |
| Short versus long | For adjusted close and synthetic adjusted volume, compare rolling means over 1 vs 10, 10 vs 60, and 60 vs 240 sessions. Also compare rolling standard deviations over 10 vs 60 and 60 vs 240 sessions. Divide both sides by the series' current level before comparison. | (1, 10), (10, 60), (60, 240) | 10 |
| Close versus other prices | Compare adjusted close separately with adjusted open, high, and low. For each pair, compare rolling means; for windows longer than one session, also compare rolling standard deviations. Divide both sides by the current adjusted close. | Mean: 1, 10, 60, 240; standard deviation: 10, 60, 240 | 21 |
| Rolling correlation | Correlate the one-session adjusted-close change with the corresponding one-session changes in adjusted open, high, low, and synthetic adjusted volume. The correlation uses changes rather than absolute price levels. | 10, 60, 240 | 12 |
| Turnover | Divide raw volume by the shares recorded in quarterly membership, provided that the selection date was already known. Calculate its rolling mean and standard deviation, then compare the means of short and long windows. Missing shares produce missing turnover features. | Mean: 1, 10, 60, 240; standard deviation: 10, 60, 240; mean comparisons: (1, 10), (10, 60), (60, 240) | 10 |

There are **86 price/volume features** in total. Each "A versus B" feature uses the same transformation:

`sign(A) * log(1 + abs(A)) - sign(B) * log(1 + abs(B))`.

For price-level comparisons, `A` and `B` are normalized by the **same current adjusted close**. For adjusted-volume comparisons, both are normalized by the same current adjusted volume. This preserves their relative difference without making the feature primarily reflect a stock's absolute price or volume scale. Turnover is already a ratio and needs no price normalization.

A rolling feature is missing when its required observations are unavailable. We do not fill those gaps here. Quarterly recorded shares can also become less representative after a corporate action; turnover should therefore be interpreted as an estimate based on the available membership record.

In [14]:
# Build each stock's rolling features on the complete market-session calendar.
WINDOWS = (1, 10, 60, 240)
sessions = pd.DatetimeIndex(calendar_df["CAL_date"])

def signed_log(x):
    return np.sign(x) * np.log1p(np.abs(x))

parts, adjustment_preview = [], None
for symbol, rows in stock_df.groupby("PRICE_symbol", sort=False):
    g = rows.set_index("PRICE_date").reindex(sessions)
    raw_close, adjusted_close = g["PRICE_close"], g["PRICE_adj_close"]
    factor = (adjusted_close / raw_close).where(raw_close.gt(0) & adjusted_close.gt(0))
    factor = factor.where(np.isfinite(factor) & factor.gt(0))

    level = {"close": adjusted_close.where(adjusted_close.gt(0))}
    for name in ("open", "high", "low"):
        x = g[f"PRICE_{name}"]
        level[name] = (x * factor).where(x.gt(0))
    raw_volume = g["PRICE_volume"]
    level["volume"] = (raw_volume / factor).where(raw_volume.ge(0) & np.isfinite(raw_volume))

    # A missing previous market session produces a missing one-session change.
    ret = {k: s.pct_change(fill_method=None).replace([np.inf, -np.inf], np.nan)
           for k, s in level.items()}
    mean = {k: {t: s.rolling(t, min_periods=t).mean() for t in WINDOWS}
            for k, s in level.items()}
    std = {k: {t: s.rolling(t, min_periods=t).std() for t in WINDOWS[1:]}
           for k, s in level.items()}
    f = {}

    # Distribution of adjusted-close and synthetic adjusted-volume changes.
    for name in ("close", "volume"):
        for t in WINDOWS:
            window = ret[name].rolling(t, min_periods=t)
            f[f"PV_{name}_ret_mean_{t}"] = window.mean()
            if t > 1:
                f[f"PV_{name}_ret_std_{t}"] = window.std()
                f[f"PV_{name}_ret_skew_{t}"] = window.skew()
                f[f"PV_{name}_ret_kurt_{t}"] = window.kurt()

    # Current drawdown, simple-window RSI, and price-normalized MACD.
    for t in WINDOWS[1:]:
        f[f"PV_drawdown_{t}"] = level["close"] / level["close"].rolling(t, min_periods=t).max() - 1
    delta = level["close"].diff()
    for t in (10, 60):
        gain = delta.clip(lower=0).rolling(t, min_periods=t).mean()
        loss = (-delta).clip(lower=0).rolling(t, min_periods=t).mean()
        total = gain + loss
        f[f"PV_rsi_{t}"] = (100 * gain / total).where(total.ne(0), 50)
    ema12 = level["close"].ewm(span=12, adjust=False, min_periods=12).mean()
    ema26 = level["close"].ewm(span=26, adjust=False, min_periods=26).mean()
    macd = ema12 - ema26
    signal = macd.ewm(span=9, adjust=False, min_periods=9).mean()
    recent_quotes = level["close"].rolling(35, min_periods=35).count().eq(35)
    f["PV_macd"] = (macd / level["close"]).where(recent_quotes)
    f["PV_macd_hist"] = ((macd - signal) / level["close"]).where(recent_quotes)

    # A vs B: short versus long windows for each series.
    for name in ("close", "volume"):
        base = level[name].where(level[name].gt(0))
        for short, long in ((1, 10), (10, 60), (60, 240)):
            f[f"PV_{name}_mean_{short}_{long}"] = (
                signed_log(mean[name][short] / base) - signed_log(mean[name][long] / base)
            )
            if short > 1:
                f[f"PV_{name}_std_{short}_{long}"] = (
                    signed_log(std[name][short] / base) - signed_log(std[name][long] / base)
                )

    # A vs B: adjusted close against adjusted open, high, and low.
    for name in ("open", "high", "low"):
        for t in WINDOWS:
            f[f"PV_close_{name}_mean_{t}"] = (
                signed_log(mean["close"][t] / level["close"])
                - signed_log(mean[name][t] / level["close"])
            )
            if t > 1:
                f[f"PV_close_{name}_std_{t}"] = (
                    signed_log(std["close"][t] / level["close"])
                    - signed_log(std[name][t] / level["close"])
                )

    # Rolling correlations use one-session changes, rather than price levels.
    for name in ("open", "high", "low", "volume"):
        for t in WINDOWS[1:]:
            f[f"PV_corr_close_{name}_{t}"] = ret["close"].rolling(t, min_periods=t).corr(ret[name])

    # Use shares only after their quarterly selection date is known.
    known_shares = g["MEM_selection_shares"].where(
        g["MEM_selection_date"].le(g.index.to_series())
    )
    turnover = (raw_volume / known_shares).where(raw_volume.ge(0) & known_shares.gt(0))
    turnover_mean = {t: turnover.rolling(t, min_periods=t).mean() for t in WINDOWS}
    for t in WINDOWS:
        f[f"PV_turnover_mean_{t}"] = turnover_mean[t]
        if t > 1:
            f[f"PV_turnover_std_{t}"] = turnover.rolling(t, min_periods=t).std()
    for short, long in ((1, 10), (10, 60), (60, 240)):
        f[f"PV_turnover_mean_{short}_{long}"] = (
            signed_log(turnover_mean[short]) - signed_log(turnover_mean[long])
        )

    # Keep historical nontradeable rows for calculation; retain eligible rows only in the result.
    eligible = g["flag_tradeable"].eq(1)
    if adjustment_preview is None and eligible.any():
        adjustment_preview = pd.DataFrame({
            "raw_close": raw_close, "adj_close": level["close"], "factor": factor,
            "adj_open": level["open"], "adj_high": level["high"], "adj_low": level["low"],
            "raw_volume": raw_volume, "adjusted_volume": level["volume"]
        }).loc[eligible].head(3)
    if eligible.any():
        metadata = ["ENTRY_date", "scheduled_exit", "label_end", "target",
                    "FUN_available_date", "FUN_filing_age_days",
                    "FUN_fiscal_age_days", "UNIV_sector"]
        chunk = pd.concat([
            g.loc[eligible, metadata],
            pd.DataFrame(f, index=sessions).replace([np.inf, -np.inf], np.nan)
              .loc[eligible].astype("float32")
        ], axis=1)
        chunk.insert(0, "PRICE_date", chunk.index)
        chunk.insert(0, "PRICE_symbol", symbol)
        parts.append(chunk.reset_index(drop=True))

feature_df = pd.concat(parts, ignore_index=True)
del parts
print("Price/volume features:", sum(c.startswith("PV_") for c in feature_df.columns))
display(adjustment_preview)
display(feature_df.head(3))

Price/volume features: 86


,raw_close,adj_close,factor,adj_open,adj_high,adj_low,raw_volume,adjusted_volume
CAL_date,,,,,,,,
2000-03-31,70.07,58.21,0.830741,59.331500,59.331500,50.376115,3963457,4.770992e+06
2000-04-03,66.03,54.86,0.830834,57.659912,57.801154,51.362187,2156125,2.595132e+06
2000-04-04,63.00,52.34,0.830794,54.857305,57.308146,45.901349,3903643,4.698691e+06


,PRICE_symbol,PRICE_date,ENTRY_date,scheduled_exit,label_end,target,FUN_available_date,FUN_filing_age_days,FUN_fiscal_age_days,UNIV_sector,...,PV_turnover_mean_1,PV_turnover_mean_10,PV_turnover_std_10,PV_turnover_mean_60,PV_turnover_std_60,PV_turnover_mean_240,PV_turnover_std_240,PV_turnover_mean_1_10,PV_turnover_mean_10_60,PV_turnover_mean_60_240
0,A,2000-03-31,2000-04-03,2000-06-28,2000-06-28,-0.183194,2000-03-16,16.0,60.0,Healthcare,...,0.009008,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,A,2000-04-03,2000-04-04,2000-06-29,2000-06-29,-0.188575,2000-03-16,19.0,63.0,Healthcare,...,0.004900,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,A,2000-04-04,2000-04-05,2000-06-30,2000-06-30,-0.239219,2000-03-16,20.0,64.0,Healthcare,...,0.008872,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


### 3.2 Fundamental features

The data-preparation step has already matched each signal with the latest quarterly statement available at that signal close. Feature engineering starts from those filing events; it does not choose a statement using a future filing date.

Because reported currency is frequently missing, we do not use raw monetary amounts as model features. Instead, each ratio combines values from the **same statement**. A denominator must be positive; otherwise the corresponding feature is missing.

| Feature | Formula | Interpretation |
|---|---|---|
| `FUN_gross_margin` | `gross_profit / revenue` | Gross profitability relative to sales |
| `FUN_operating_margin` | `operating_income / revenue` | Operating profitability relative to sales |
| `FUN_net_margin` | `net_income / revenue` | Net profitability relative to sales |
| `FUN_cashflow_margin` | `operating_cash_flow / revenue` | Operating cash generation relative to sales |
| `FUN_fcf_margin` | `free_cash_flow / revenue` | Free cash flow relative to sales |
| `FUN_operating_roa` | `operating_income / total_assets` | Quarterly operating income relative to reported assets |
| `FUN_debt_assets` | `total_debt / total_assets` | Debt relative to assets |
| `FUN_equity_assets` | `total_equity / total_assets` | Equity relative to assets |
| `FUN_cash_assets` | `cash_and_equivalents / total_assets` | Cash relative to assets |
| `FUN_current_ratio` | `total_current_assets / total_current_liabilities` | Short-term asset coverage of current liabilities |
| `FUN_accruals_assets` | `(net_income - operating_cash_flow) / total_assets` | Difference between earnings and operating cash flow, scaled by assets |

We also measure changes in five ratios: `gross_margin`, `operating_margin`, `net_margin`, `cashflow_margin`, and `debt_assets`. Each `FUN_qoq_*` feature equals the **current ratio minus the previous distinct fiscal quarter's ratio**. It is a difference between dimensionless ratios, not a percentage change in monetary amounts. We require fiscal period ends approximately one quarter apart (70–115 days). A subsequent filing for the same fiscal quarter does not count as another quarter.

Two previously calculated age features are retained:

- `FUN_filing_age_days`: calendar days from the matched filing date to the signal date.
- `FUN_fiscal_age_days`: calendar days from the matched fiscal period end to the signal date.

A statement more than 550 calendar days past its fiscal period end is considered stale: its derived ratios and quarter-to-quarter changes become missing, while its age remains observable. No other missing financial feature is filled at this stage.

This gives **11 current-statement ratios + 5 changes from the previous quarter + 2 age features = 18 fundamental features**.

In [15]:
# Derive dimensionless features at the filing-event level, before joining signal rows.
financial_events = financial_df.sort_values(
    ["FUN_symbol", "FUN_available_date", "FUN_fiscal_date"]
).copy()
fin_features = financial_events[
    ["FUN_symbol", "FUN_fiscal_date", "FUN_available_date"]
].copy()

ratios = {
    "gross_margin": ("FUN_gross_profit", "FUN_revenue"),
    "operating_margin": ("FUN_operating_income", "FUN_revenue"),
    "net_margin": ("FUN_net_income", "FUN_revenue"),
    "cashflow_margin": ("FUN_operating_cash_flow", "FUN_revenue"),
    "fcf_margin": ("FUN_free_cash_flow", "FUN_revenue"),
    "operating_roa": ("FUN_operating_income", "FUN_total_assets"),
    "debt_assets": ("FUN_total_debt", "FUN_total_assets"),
    "equity_assets": ("FUN_total_equity", "FUN_total_assets"),
    "cash_assets": ("FUN_cash_and_equivalents", "FUN_total_assets"),
    "current_ratio": ("FUN_total_current_assets", "FUN_total_current_liabilities"),
}
for name, (numerator, denominator) in ratios.items():
    d = financial_events[denominator].where(financial_events[denominator].gt(0))
    fin_features[f"FUN_{name}"] = financial_events[numerator] / d

assets = financial_events["FUN_total_assets"].where(
    financial_events["FUN_total_assets"].gt(0)
)
fin_features["FUN_accruals_assets"] = (
    financial_events["FUN_net_income"]
    - financial_events["FUN_operating_cash_flow"]
) / assets
ratio_cols = [f"FUN_{name}" for name in ratios] + ["FUN_accruals_assets"]
fin_features[ratio_cols] = fin_features[ratio_cols].replace([np.inf, -np.inf], np.nan)

# Find the preceding distinct fiscal quarter; use its first available filing.
first_fiscal = fin_features.drop_duplicates(
    ["FUN_symbol", "FUN_fiscal_date"], keep="first"
).sort_values(["FUN_symbol", "FUN_fiscal_date"]).copy()
first_fiscal["previous_fiscal"] = first_fiscal.groupby(
    "FUN_symbol"
)["FUN_fiscal_date"].shift()
valid_previous = (
    first_fiscal["FUN_fiscal_date"] - first_fiscal["previous_fiscal"]
).dt.days.between(70, 115)

change_names = ("gross_margin", "operating_margin", "net_margin",
                "cashflow_margin", "debt_assets")
for name in change_names:
    first_fiscal[f"previous_{name}"] = first_fiscal.groupby("FUN_symbol")[
        f"FUN_{name}"
    ].shift().where(valid_previous)

previous_cols = [f"previous_{name}" for name in change_names]
fin_features = fin_features.merge(
    first_fiscal[["FUN_symbol", "FUN_fiscal_date"] + previous_cols],
    on=["FUN_symbol", "FUN_fiscal_date"], how="left", validate="many_to_one"
)
for name in change_names:
    fin_features[f"FUN_qoq_{name}"] = (
        fin_features[f"FUN_{name}"] - fin_features.pop(f"previous_{name}")
    )
change_cols = [f"FUN_qoq_{name}" for name in change_names]
fin_features = fin_features.drop(columns="FUN_fiscal_date")

# Match the same filing event already selected in stock_df.
feature_df["FUN_available_date"] = feature_df["FUN_available_date"].astype("datetime64[ns]")
fin_features["FUN_available_date"] = fin_features["FUN_available_date"].astype("datetime64[ns]")
feature_df = feature_df.merge(
    fin_features,
    left_on=["PRICE_symbol", "FUN_available_date"],
    right_on=["FUN_symbol", "FUN_available_date"],
    how="left", validate="many_to_one"
).drop(columns="FUN_symbol")

# Old statements no longer supply ratios; their age remains observable.
feature_df.loc[
    feature_df["FUN_fiscal_age_days"].gt(550), ratio_cols + change_cols
] = np.nan
feature_df[ratio_cols + change_cols] = feature_df[
    ratio_cols + change_cols
].astype("float32")

print("Financial features:", len(ratio_cols + change_cols) + 2)
display(feature_df[
    ["PRICE_symbol", "PRICE_date", "FUN_available_date",
     "FUN_filing_age_days", "FUN_fiscal_age_days",
     "FUN_net_margin", "FUN_qoq_net_margin"]
].dropna(subset=["FUN_net_margin"]).head(3))

Financial features: 18


,PRICE_symbol,PRICE_date,FUN_available_date,FUN_filing_age_days,FUN_fiscal_age_days,FUN_net_margin,FUN_qoq_net_margin
0,A,2000-03-31,2000-03-16,16.0,60.0,0.058326,NaN
1,A,2000-04-03,2000-03-16,19.0,63.0,0.058326,NaN
2,A,2000-04-04,2000-03-16,20.0,64.0,0.058326,NaN


### 3.3 Sector features and final panel

`UNIV_sector` is converted into one-hot columns named `SECTOR_*`. Each stock has one active sector column. A missing sector is represented by `SECTOR_Unknown`. We use sector only; industry, country, and exchange are not included in the model features. The supplied sector is a snapshot descriptor, so it may not perfectly describe a stock's historical sector.

The final `feature_df` contains only rows that had `flag_tradeable = 1` in `stock_df`. We keep all historical rows while calculating rolling price features and apply this filter only when assembling the final feature rows. A tradeable signal remains in `feature_df` even when its future target has not yet been observed.

| Column or group | Purpose | Model input? |
|---|---|---|
| `PRICE_symbol` | Stock identifier | No |
| `PRICE_date` | Signal date; today's close is already known | No |
| `ENTRY_date` | Planned next-market-session entry date | No |
| `scheduled_exit` | Exit scheduled under the current 60-market-session holding convention | No |
| `label_end` | Date of the observed quote used for the exit label; may be missing | No |
| `FUN_available_date` | Availability date of the matched financial statement | No |
| `target` | Realized adjusted-price return from the observed entry to the observed exit; may be missing | No |
| `PV_*` | The 86 price and volume features | Yes |
| `FUN_*` age and ratio columns | The 18 financial features | Yes |
| `SECTOR_*` | One-hot sector features; number depends on the sectors present | Yes |

For later modeling, use `feature_df[feature_cols]` as the inputs. Use `target` only as the supervised-learning outcome, and select rows with observed targets when fitting or validating a model. In particular, `label_end`, `scheduled_exit`, and `target` must never enter the model inputs. Feature missing values remain missing until the model-training pipeline specifies how to handle them.

In [16]:
# Replace the sector text with one-hot feature columns.
sector_features = pd.get_dummies(
    feature_df.pop("UNIV_sector").fillna("Unknown"),
    prefix="SECTOR", dtype="uint8"
)
for col in sector_features:
    feature_df[col] = sector_features[col]

# Separate the columns allowed into a model from dates, identifiers, and target.
meta_cols = ["PRICE_symbol", "PRICE_date", "ENTRY_date", "scheduled_exit",
             "label_end", "FUN_available_date", "target"]
feature_cols = [
    col for col in feature_df
    if col.startswith(("PV_", "FUN_", "SECTOR_")) and col != "FUN_available_date"
]
unexpected = sorted(set(feature_df) - set(meta_cols) - set(feature_cols))
if unexpected:
    raise ValueError(f"Unexpected final columns: {unexpected}")

display(pd.Series({
    "tradeable_signal_rows": len(feature_df),
    "observed_targets": int(feature_df["target"].notna().sum()),
    "targets_not_yet_observed": int(feature_df["target"].isna().sum()),
    "price_volume_features": sum(c.startswith("PV_") for c in feature_cols),
    "fundamental_features": sum(c.startswith("FUN_") for c in feature_cols),
    "sector_features": sum(c.startswith("SECTOR_") for c in feature_cols),
}, name="count").to_frame())

display(feature_df[
    meta_cols + ["PV_close_ret_mean_10", "PV_turnover_mean_10",
                 "FUN_net_margin"] + sector_features.columns[:1].tolist()
].head())

,count
tradeable_signal_rows,12284888
observed_targets,12120871
targets_not_yet_observed,164017
price_volume_features,86
fundamental_features,18
sector_features,11


,PRICE_symbol,PRICE_date,ENTRY_date,scheduled_exit,label_end,FUN_available_date,target,PV_close_ret_mean_10,PV_turnover_mean_10,FUN_net_margin,SECTOR_Basic Materials
0,A,2000-03-31,2000-04-03,2000-06-28,2000-06-28,2000-03-16,-0.183194,-0.014264,NaN,0.058326,0
1,A,2000-04-03,2000-04-04,2000-06-29,2000-06-29,2000-03-16,-0.188575,-0.014038,NaN,0.058326,0
2,A,2000-04-04,2000-04-05,2000-06-30,2000-06-30,2000-03-16,-0.239219,-0.018192,NaN,0.058326,0
3,A,2000-04-05,2000-04-06,2000-07-03,2000-07-03,2000-03-16,-0.298111,-0.019982,NaN,0.058326,0
4,A,2000-04-06,2000-04-07,2000-07-05,2000-07-05,2000-03-16,-0.430937,-0.011880,NaN,0.058326,0


### 3.4 Select signals with a complete planned holding period

The backtest ends on September 30, 2026. Under this experiment's 60-market-session holding rule, we do not open a new position unless its **scheduled** exit falls on or before that fixed end date. The market calendar and test end are known when the signal is formed, so this restriction does not depend on a future stock quote.

We also require sufficient information available at the signal close. For each stock, the analysis starts after its 240-session adjusted-close, adjusted-volume, and turnover windows first become complete and after two approximately consecutive quarterly statements have become available. A later signal is excluded if any of those three core 240-session windows becomes incomplete again.

These rules define the signal universe. They do **not** inspect `target` or the future observed exit quote. A scheduled exit may still lack a valid quote, and a next-session entry may fail because that day's quote is unavailable. The backtester must handle those events when they occur rather than removing the stock using hindsight.

We retain only signals satisfying the fixed exit horizon and the historical-information requirements in `feature_df`. A separate labeled mask identifies rows that can be used for model evaluation; at each model fit, the label must also have finished before the fit date.

In [17]:
# Use the fixed backtest end, not the stock's future observed exit price.
TEST_END = pd.Timestamp("2026-09-30")
warmup_cols = [
    "PV_close_ret_mean_240",
    "PV_volume_ret_mean_240",
    "PV_turnover_mean_240",
]

# Find each stock's first signal with complete core 240-session history.
window_complete = feature_df[warmup_cols].notna().all(axis=1)
price_ready = feature_df["PRICE_date"].where(window_complete).groupby(
    feature_df["PRICE_symbol"]
).min().rename("price_volume_ready")

# Find when two distinct, approximately consecutive quarterly reports became available.
filings = financial_df.sort_values(
    ["FUN_symbol", "FUN_available_date", "FUN_fiscal_date"]
).drop_duplicates(
    ["FUN_symbol", "FUN_fiscal_date"], keep="first"
).sort_values(["FUN_symbol", "FUN_fiscal_date"])

previous_fiscal = filings.groupby("FUN_symbol")["FUN_fiscal_date"].shift()
quarter_gap = (filings["FUN_fiscal_date"] - previous_fiscal).dt.days
financial_ready = filings.loc[
    quarter_gap.between(70, 115)
].groupby("FUN_symbol")["FUN_available_date"].min().rename("financial_ready")

# A stock becomes eligible on the later of its two readiness dates.
start_dates = pd.concat([price_ready, financial_ready], axis=1)
both_ready = start_dates[["price_volume_ready", "financial_ready"]].notna().all(axis=1)
start_dates["analysis_start"] = start_dates[
    ["price_volume_ready", "financial_ready"]
].max(axis=1).where(both_ready)

# Every condition here is determined without observing the future stock return.
exit_in_test = (
    feature_df["scheduled_exit"].notna()
    & feature_df["scheduled_exit"].le(TEST_END)
)
after_start = feature_df["PRICE_date"].ge(
    feature_df["PRICE_symbol"].map(start_dates["analysis_start"])
)
signal_mask = exit_in_test & after_start & window_complete

# Report mutually exclusive removal reasons before filtering.
removals = pd.Series({
    "original_tradeable_signals": len(feature_df),
    "removed_exit_after_test_end": int((~exit_in_test).sum()),
    "removed_before_ticker_start": int((exit_in_test & ~after_start).sum()),
    "removed_later_window_gap": int(
        (exit_in_test & after_start & ~window_complete).sum()
    ),
    "signals_retained": int(signal_mask.sum()),
}, name="rows")
display(removals.to_frame())

# The resulting feature_df is the candidate panel for later predictions.
feature_df = feature_df.loc[signal_mask].reset_index(drop=True)
labeled_mask = feature_df["target"].notna()

print("Signal panel:", feature_df.shape)
print("Stocks:", feature_df["PRICE_symbol"].nunique())
print("Signals with an observed target:", int(labeled_mask.sum()))
print("Scheduled exits without an observed target:", int((~labeled_mask).sum()))

,rows
original_tradeable_signals,12284888
removed_exit_after_test_end,118891
removed_before_ticker_start,1585509
removed_later_window_gap,504412
signals_retained,10076076


Signal panel: (10076076, 122)
Stocks: 3262
Signals with an observed target: 10048682
Scheduled exits without an observed target: 27394


### 3.5 Missingness and daily stock coverage

Missingness is measured across the retained signal universe. We also show missingness among rows with an observed target, since that is the subset on which a return-prediction model can be evaluated. A missing target does not remove a stock from the signal universe.

We count distinct usable stocks on every market session between the first and last retained signal dates, including sessions with zero eligible stocks in the daily average. We report the earliest and latest signal dates, the number of stocks on each of those dates, and the average number of stocks per market session. A per-stock table records each stock's first and last retained signal dates.

When fitting a model at a particular date, we will additionally require `label_end` to precede that fit date. This prevents a return that finishes later from entering earlier training data.

In [18]:
# Compare missingness for all retained signals and the observed-label subset.
n_signals, n_labeled = len(feature_df), int(labeled_mask.sum())
if n_signals == 0 or n_labeled == 0:
    raise ValueError("No signal or labeled rows remain")

rows = []
for col in feature_df.columns:
    missing = feature_df[col].isna()
    rows.append({
        "column": col,
        "role": "feature" if col in feature_cols else "metadata/target",
        "missing_pct_signals": round(100 * missing.sum() / n_signals, 2),
        "missing_pct_labeled": round(100 * (missing & labeled_mask).sum() / n_labeled, 2),
    })
missing_report = pd.DataFrame(rows)

with pd.option_context("display.max_rows", None):
    display(missing_report.sort_values(
        ["missing_pct_signals", "column"], ascending=[False, True]
    ).reset_index(drop=True))

# Summarize stock counts for both the signal universe and its labeled subset.
coverage_rows = []
daily_stock_counts = {}
market_calendar = pd.DatetimeIndex(calendar_df["CAL_date"])

for name, mask in [
    ("signals", pd.Series(True, index=feature_df.index)),
    ("observed_labels", labeled_mask),
]:
    observed = feature_df.loc[mask].groupby("PRICE_date")["PRICE_symbol"].nunique()
    first_date, last_date = observed.index.min(), observed.index.max()
    dates = market_calendar[
        (market_calendar >= first_date) & (market_calendar <= last_date)
    ]
    counts = observed.reindex(dates, fill_value=0).rename("usable_stocks")
    daily_stock_counts[name] = counts
    coverage_rows.append({
        "sample": name,
        "first_signal": first_date,
        "stocks_on_first": int(counts.loc[first_date]),
        "last_signal": last_date,
        "stocks_on_last": int(counts.loc[last_date]),
        "market_sessions": len(counts),
        "mean_stocks_per_session": round(counts.mean(), 2),
    })

display(pd.DataFrame(coverage_rows).set_index("sample"))

# Inspect individual stocks' first and last available signal dates.
ticker_ranges = feature_df.groupby("PRICE_symbol").agg(
    first_signal=("PRICE_date", "min"),
    last_signal=("PRICE_date", "max"),
    signal_rows=("PRICE_date", "size"),
).join(start_dates)
display(ticker_ranges.sort_values("first_signal").head(10))

,column,role,missing_pct_signals,missing_pct_labeled
0,FUN_current_ratio,feature,4.49,4.49
1,FUN_qoq_cashflow_margin,feature,2.77,2.77
2,FUN_qoq_gross_margin,feature,2.44,2.43
3,FUN_qoq_operating_margin,feature,2.44,2.43
4,FUN_qoq_net_margin,feature,2.43,2.43
5,FUN_cashflow_margin,feature,1.40,1.39
6,FUN_fcf_margin,feature,1.40,1.39
7,FUN_qoq_debt_assets,feature,1.26,1.26
8,FUN_gross_margin,feature,1.19,1.19
9,FUN_net_margin,feature,1.19,1.18


,first_signal,stocks_on_first,last_signal,stocks_on_last,market_sessions,mean_stocks_per_session
sample,,,,,,
signals,2000-12-13,854,2026-07-06,1730,6425,1568.26
observed_labels,2000-12-13,854,2026-07-06,1710,6425,1564.00


,first_signal,last_signal,signal_rows,price_volume_ready,financial_ready,analysis_start
PRICE_symbol,,,,,,
KFY,2000-12-13,2026-07-06,6425,2000-12-13,2000-08-01,2000-12-13
LANC,2000-12-13,2025-06-30,6171,2000-12-13,2000-09-27,2000-12-13
LAMR,2000-12-13,2026-07-06,6425,2000-12-13,2000-08-12,2000-12-13
L,2000-12-13,2026-07-06,6425,2000-12-13,2000-08-08,2000-12-13
CDZI,2000-12-13,2014-03-28,2394,2000-12-13,2000-08-16,2000-12-13
KTOS,2000-12-13,2026-07-06,5997,2000-12-13,2000-08-15,2000-12-13
KSS,2000-12-13,2026-07-06,6425,2000-12-13,2000-06-06,2000-12-13
KSPN,2000-12-13,2014-02-18,2165,2000-12-13,2000-06-14,2000-12-13
KRC,2000-12-13,2026-07-06,6425,2000-12-13,2000-08-15,2000-12-13


## 4 Save the Data

In [19]:
# Save the prepared panel and benchmark without modifying the source database.
save_path = db_path.parent / "hw4_yisu_60_sessions.duckdb"
save = duckdb.connect(str(save_path))
try:
    save.register("feature_input", feature_df)
    save.register("benchmark_input", benchmark_df)
    save.register("calendar_input", calendar_df[["CAL_date"]])

    save.execute("BEGIN TRANSACTION")
    save.execute("CREATE OR REPLACE TABLE features AS SELECT * FROM feature_input")
    save.execute("CREATE OR REPLACE TABLE benchmark AS SELECT * FROM benchmark_input")
    save.execute("CREATE OR REPLACE TABLE calendar AS SELECT * FROM calendar_input")
    save.execute("""
        CREATE OR REPLACE VIEW sample_masks AS
        SELECT PRICE_symbol, PRICE_date,
               TRUE AS signal_eligible,
               target IS NOT NULL AND label_end IS NOT NULL AS label_observed
        FROM features
    """)
    save.execute("COMMIT")

    print("Saved to:", save_path)
    display(save.execute("""
        SELECT COUNT(*) AS signal_rows,
               COUNT(*) FILTER (WHERE label_observed) AS rows_with_label
        FROM sample_masks
    """).df())
    display(save.execute("""
        SELECT 'features' AS table_name, COUNT(*) AS rows FROM features
        UNION ALL SELECT 'benchmark', COUNT(*) FROM benchmark
        UNION ALL SELECT 'calendar', COUNT(*) FROM calendar
    """).df())
except Exception:
    save.execute("ROLLBACK")
    raise
finally:
    save.close()

Saved to: /Users/wangyisu/Desktop/Documents/pre-courses/Python for Financial Engineers/UCB-MFE-python-preprogram-2026-winter/Lectures/modeling/data/hw4_yisu_60_sessions.duckdb


,signal_rows,rows_with_label
0,10076076,10048682


,table_name,rows
0,features,10076076
1,benchmark,6726
2,calendar,6726
